# Pdac Nact

In-house PDAC T2 reference to T3 query, using the original prepared exports. Source: `MPNN_SelfDistill/benchmark/NACT/exp2_all.ipynb`. Only the original training and metric steps are retained; figure generation is omitted. Package imports, extracted-data paths, run output paths, and the CUDA device are the only training-cell adaptations.

Original training configuration: GNN=gcn, batch_size=512, max_steps=500, high=False, rate=1.

Extract the matching restricted Zenodo archive into `data/experiments/` before running. Use `SCPATH_STUDY_DATA` to point to a different extraction directory and `SCPATH_DEVICE` to select a GPU. Set `PYTHONHASHSEED=42` before starting Jupyter, because the source preprocessing uses set-based gene intersections. Run cells in order with the `scPath` kernel. Historical result files in the archive are for comparison only; this notebook writes new outputs under `output/experiments/`.


In [ ]:
import os
import sys
from pathlib import Path
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scpath/models.py").is_file())
sys.path.insert(0, str(repo_root))
data_root = Path(os.environ.get("SCPATH_STUDY_DATA", str(repo_root / "data/experiments")))
study_data = data_root / "pdac_nact/original_paths/data/10x_export"
run_dir = repo_root / "output/experiments/pdac_nact"
if not study_data.is_dir():
    raise FileNotFoundError(f"Extract the pdac_nact Zenodo archive under {data_root}")
run_dir.mkdir(parents=True, exist_ok=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this historical training workflow")
print("GPU:", torch.cuda.get_device_name(torch.device(os.environ.get("SCPATH_DEVICE", "cuda:0"))))


In [ ]:
import sys
sys.path.insert(0, str(repo_root))
from scpath.utils.data_preprocess import data_processing
from scpath.utils.pathway import built_pathway_graphs
from scpath.utils.data_preprocess import preprocess
from scpath.train import train
import torch
import os
from scpath.config import Config
import pandas as pd
import scipy.sparse as sparse
from scipy.io import mmread, mmwrite

train_path = str(study_data / "t2") + "/"  
test_path = str(study_data / "t3") + "/"  
path_output = str(run_dir / "output") + "/"
species_train='human'
species_test='human'
plot=False
save_data=False
os.makedirs(path_output, exist_ok=True)

adata_train_rna,adata_test_rna=data_processing(train_path,test_path,path_output,species_train,species_test,plot,save_data,qc_filter_bool=False)

n_train = adata_train_rna.n_obs
n_test = adata_test_rna.n_obs
total_samples = n_train + n_test
    

gnn_type='gcn'
batch_size=512
max_epochs=500
print(f"Training strategy: GNN={gnn_type} | batch={batch_size} | epochs={max_epochs}")
    

effective_steps = max_epochs

pathway_names=['Reactome','KEGG']  
file_path = str(repo_root / "data/pathway/human") + "/"  
built_pathway_graphs(file_path,pathway_names,adata_train_rna,adata_test_rna,path_output,n_neighbors=5)


adata_train_rna, hvg_genes = preprocess(adata_train_rna)
adata_test_rna = preprocess(adata_test_rna, hvg_genes=hvg_genes)

device = torch.device(os.environ.get("SCPATH_DEVICE", "cuda:0"))
config = Config(str(run_dir) + "/",adata_train_rna.X.shape[1],device,False,batch_size=int(batch_size),gnn_type=gnn_type,max_steps=effective_steps,K=len(adata_test_rna.obs['cell_type'].unique()),rate=1)
print(f"batch_size type: {type(config.batch_size)}")
train(config,adata_train_rna,adata_test_rna,pathway_names)



In [ ]:
import scanpy as sc
adata_test_result = sc.read_h5ad(run_dir / "logs/test_results_distilled.h5ad")
print("Query label distribution:")
print(adata_test_result.obs["labels"].value_counts())
print("Final predicted class distribution:")
print(adata_test_result.obs["final_distill_preds"].value_counts())
print("The historical PDAC query has only an unseen/private label and unknown cell types; ordinary accuracy and macro-F1 against that placeholder are not meaningful.")
